# SAP HANA Cloud Memory Store for LangGraph

A LangGraph [`BaseStore`](https://langchain-ai.github.io/langgraph/reference/store/) holds **long-term, cross-thread memory** — data that outlives a single conversation and is shared across threads and users (user preferences, facts, documents, agent memories).

`HanaStore` (and its native-async twin `AsyncHanaStore`) back that store with **SAP HANA Cloud**. It provides:
- **hierarchical key-value storage** — items addressed by a *namespace* tuple and a *key*,
- **metadata search** — filter and paginate over stored JSON values,
- optional **TTL** — items expire and are swept automatically,
- optional **semantic / vector search** — rank items by meaning, using either an external embedding model or HANA's **in-database** embeddings.

## Setup & installation

Install `langchain-hana`. LangGraph is pulled in automatically as a dependency, so no
separate install is needed:

In [ ]:
!pip install langchain-hana

Open a connection to your SAP HANA Cloud instance with `hdbcli`.

In [2]:
import os

from dotenv import load_dotenv
from hdbcli import dbapi

# Load HANA_DB_ADDRESS / HANA_DB_PORT / HANA_DB_USER / HANA_DB_PASSWORD from a .env
load_dotenv()

connection = dbapi.connect(
    address=os.environ.get("HANA_DB_ADDRESS"),
    port=os.environ.get("HANA_DB_PORT"),
    user=os.environ.get("HANA_DB_USER"),
    password=os.environ.get("HANA_DB_PASSWORD"),
)

## Creating a store

Pass the connection to `HanaStore`. Like the checkpointer, the constructor is self-initializing (it applies the idempotent schema migrations) and puts the connection in manual-commit mode so each batch commits atomically.

You can override `schema_name` and the table names; semantic search (`index`) and `ttl` are covered further below.

In [3]:
from langchain_hana import HanaStore

store = HanaStore(connection)  # plain key-value store, no TTL, no vector index

## Core key-value operations

Items live under a **namespace** — a tuple of strings forming a hierarchy — and a **key**. Values are JSON-serializable dicts.

In [4]:
namespace = ("users", "alice")

store.put(namespace, "profile", {"name": "Alice", "theme": "dark"})

item = store.get(namespace, "profile")
print(item.value)  # {'name': 'Alice', 'theme': 'dark'}
print(item.namespace)  # ('users', 'alice')
print(item.key)  # 'profile'
print(item.created_at, item.updated_at)

store.delete(namespace, "profile")
print(store.get(namespace, "profile"))  # None

{'name': 'Alice', 'theme': 'dark'}
('users', 'alice')
profile
2026-11-20 13:39:54.373000 2026-11-20 13:39:54.373000
None


## Search: metadata filtering and pagination

`search` returns items under a namespace **prefix**. Without a `query` it filters and paginates by stored metadata. Operators such as `$gte`, `$ne`, `$in` are supported in filters.

In [5]:
ns = ("docs",)
store.put(ns, "a", {"color": "red", "score": 1})
store.put(ns, "b", {"color": "red", "score": 5})
store.put(ns, "c", {"color": "blue", "score": 9})

reds = store.search(ns, filter={"color": "red"})
print({i.key for i in reds})  # {'a', 'b'}

high = store.search(ns, filter={"score": {"$gte": 5}})
print({i.key for i in high})  # {'b', 'c'}

# Pagination
page = store.search(ns, limit=1, offset=1)
print(len(page))  # 1

# An empty prefix searches across every namespace
everything = store.search(())
print(len(everything))

{'a', 'b'}
{'c', 'b'}
1
3


## Listing namespaces

`list_namespaces` enumerates the stored namespaces, with optional `prefix`/`suffix` matching and a `max_depth` to truncate the hierarchy.

In [6]:
store.put(("a", "b", "c"), "k", {"v": 1})
store.put(("a", "b", "d"), "k", {"v": 1})
store.put(("a", "x"), "k", {"v": 1})

print(store.list_namespaces(prefix=("a", "b")))  # [('a','b','c'), ('a','b','d')]
print(store.list_namespaces(prefix=("a",), max_depth=2))  # [('a','b'), ('a','x')]
print(store.list_namespaces(suffix=("c",)))  # [('a','b','c')]

[('a', 'b', 'c'), ('a', 'b', 'd')]


[('a', 'b'), ('a', 'x')]
[('a', 'b', 'c')]


## Time-to-live (TTL)

Give the store a `ttl` config to expire items automatically. `default_ttl` is in **minutes**; `refresh_on_read=True` extends an item's life whenever it is read. Expired items are hidden from reads immediately and can be physically removed with `sweep_ttl()`.

In [7]:
store = HanaStore(
    connection,
    ttl={"default_ttl": 60, "refresh_on_read": True},  # expire 60 minutes after access
)

store.put(("session",), "token", {"value": "abc"})
# Override the default per item (here ~1.2 seconds):
store.put(("session",), "short", {"value": "xyz"}, ttl=0.02)

removed = store.sweep_ttl()  # delete rows already past expiry; returns the count
print("swept:", removed)

swept: 0


## Semantic (vector) search with external embeddings

To rank items by meaning, build the store with an `index` config:

- **`dims`** — the embedding dimensionality (must match your model),
- **`embed`** — any LangChain `Embeddings` instance (or a `"provider:model"` string),
- **`fields`** — which fields of the stored value to embed (JSON paths, e.g. `["text"]`).

At `put` time the selected fields are embedded and stored; `search(..., query=...)` embeds the query and ranks by cosine similarity.

> **Dedicated tables per index.** A vectors table is created once with a **fixed dimensionality and column type**. Because the sections below use different embedding models (and vector column types), each gives its store its own `migrations_table`/`store_table`/`vectors_table` so their vector tables don't collide in the same schema. In your own app you normally pick one configuration and keep the default names.

In [8]:
# Any LangChain Embeddings implementation works, e.g. from the SAP
# generative AI hub SDK (text-embedding-3-small has dims=1536):
from gen_ai_hub.proxy.langchain.openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(proxy_model_name="text-embedding-3-small")

store = HanaStore(
    connection,
    index={"dims": 1536, "embed": embeddings, "fields": ["text"]},
    migrations_table="STORE_EXT_MIGRATIONS",
    store_table="STORE_EXT",
    vectors_table="STORE_EXT_VECTORS",
)

ns = ("memories",)
store.put(ns, "cats", {"text": "cats are small furry feline pets"})
store.put(ns, "cars", {"text": "cars are fast motor vehicles"})

results = store.search(ns, query="kitten")
for item in results:
    print(item.key, item.score)  # 'cats' ranks first

# Skip embedding for a specific item (still stored, just not vector-searchable):
store.put(ns, "note", {"text": "internal note"}, index=False)

cats 0.4686625657631471
cars 0.1573096439917757


## Semantic search with HANA **internal** embeddings

Instead of embedding in Python, HANA can compute embeddings **inside the database** with its `VECTOR_EMBEDDING` function. Pass a `HanaInternalEmbeddings` as `embed`: no data leaves HANA for embedding, and put/search issue no Python-side embedding calls.

`dims` must match the chosen model's output dimensionality (here 768, different from the 1536-dim external model above — hence its own tables).

In [9]:
from langchain_hana import HanaInternalEmbeddings

internal = HanaInternalEmbeddings(
    # a model available on your instance
    internal_embedding_model_id="SAP_NEB.20240715",
)

store = HanaStore(
    connection,
    index={"dims": 768, "embed": internal, "fields": ["text"]},
    migrations_table="STORE_INT_MIGRATIONS",
    store_table="STORE_INT",
    vectors_table="STORE_INT_VECTORS",
)

store.put(("memories",), "cats", {"text": "cats are small furry feline pets"})
print(store.search(("memories",), query="kitten"))

[Item(namespace=['memories'], key='cats', value={'text': 'cats are small furry feline pets'}, created_at='2026-11-20T13:40:00.684000', updated_at='2026-11-20T13:40:00.684000', score=0.697585114175845)]


If the embedding model is served through a **remote source** (optionally in a specific schema), configure it on the embeddings object:

In [ ]:
internal = HanaInternalEmbeddings(
    internal_embedding_model_id="<remote-model-id>",
    remote_source="<REMOTE_SOURCE_NAME>",
    remote_source_schema="<SCHEMA>",  # optional
)

### Storing half-precision vectors

By default embeddings are stored as `REAL_VECTOR` (4 bytes per element). On instances that support it you can halve the storage with `HALF_VECTOR` (2 bytes per element) via `vector_column_type` — the embedding backend (external or internal) is unchanged. Again, its own tables keep it separate from the `REAL_VECTOR` store above.

In [10]:
store = HanaStore(
    connection,
    index={"dims": 768, "embed": internal, "fields": ["text"]},
    vector_column_type="HALF_VECTOR",  # default is "REAL_VECTOR"
    migrations_table="STORE_HALF_MIGRATIONS",
    store_table="STORE_HALF",
    vectors_table="STORE_HALF_VECTORS",
)

## Using the store with an agent

Pass the store to a LangChain agent via `store=...`. Its tools can then read and write long-term memory that persists **across threads** (unlike the checkpointer's per-thread state). A tool receives the store through an `InjectedStore` annotation — it is supplied at runtime and never shown to the model.

This reuses the indexed `store` from the semantic-search section above, so the recall tool's `query=` ranks memories by similarity.

> The agent uses LangChain's `create_agent` with an LLM from the SAP generative AI hub SDK (`pip install sap-ai-sdk-gen`); any LangChain chat model works.

In [11]:
import uuid
from typing import Annotated

from gen_ai_hub.proxy.langchain.openai import ChatOpenAI
from langchain.agents import create_agent
from langgraph.prebuilt import InjectedStore
from langgraph.store.base import BaseStore

# LLM from the SAP generative AI hub SDK (sap-ai-sdk-gen)
llm = ChatOpenAI(proxy_model_name="gpt-4o", temperature=0)
# Or any LangChain chat model, e.g.:
#   from langchain_openai import ChatOpenAI; llm = ChatOpenAI(model="gpt-4o")


def save_memory(content: str, *, store: Annotated[BaseStore, InjectedStore()]) -> str:
    """Save a fact about the user to long-term memory."""
    store.put(("memories",), str(uuid.uuid4()), {"text": content})
    return "Saved."


def recall_memory(query: str, *, store: Annotated[BaseStore, InjectedStore()]) -> str:
    """Recall facts about the user from long-term memory."""
    items = store.search(("memories",), query=query, limit=3)
    return "\n".join(i.value["text"] for i in items) or "No memories yet."


agent = create_agent(
    model=llm,
    tools=[save_memory, recall_memory],
    store=store,
)

# Teach the agent something on one thread...
agent.invoke(
    {"messages": [{"role": "user", "content": "Remember that I prefer window seats."}]},
    {"configurable": {"thread_id": "thread-1"}},
)

# ...and recall it from a DIFFERENT thread: the memory lives in HANA, not the thread.
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What seat do I prefer?"}]},
    {"configurable": {"thread_id": "thread-2"}},
)
print(result["messages"][-1].content)

You prefer window seats.


## Asynchronous usage

`AsyncHanaStore` drives an `hdbcli` **async** connection. As with the async checkpointer: open the connection with `await dbapi.async_connect(...)` and call `await store.setup()` once before use (the async store does not set up in its constructor). All operations have `a`-prefixed coroutines (`aput`, `aget`, `asearch`, `adelete`, `alist_namespaces`, `asweep_ttl`).

In [12]:
from langchain_hana import AsyncHanaStore

async_connection = await dbapi.async_connect(
    address=os.environ.get("HANA_DB_ADDRESS"),
    port=os.environ.get("HANA_DB_PORT"),
    user=os.environ.get("HANA_DB_USER"),
    password=os.environ.get("HANA_DB_PASSWORD"),
)

store = AsyncHanaStore(async_connection)
await store.setup()  # required once

await store.aput(("users", "bob"), "profile", {"name": "Bob"})
item = await store.aget(("users", "bob"), "profile")
print(item.value)

{'name': 'Bob'}


## Cleanup

In [13]:
connection.close()
await async_connection.close()